In [ ]:
!java -version

[0.023s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.023s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
!python --version

Python 3.13.16


In [ ]:
%pip install pyspark==4.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813860 sha256=0a44d8bc1f175f3f349dc863f8ecffabe66336a40e519b4b8b9ef44693d21cc5
  Stored in directory: /root/.cache/pip/wheels/00/e3/92/8594f4cee2c9fd4ad82fe85e4bf2559ab8ea84ef19b1dd3d15
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.4
    Uninstalling pyspark-4.0.4:
      Successfully uninstalled pyspark-4.0.4


In [ ]:
# create spark session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.0.1
Hadoop version: 3.4.1


In [ ]:
# Install the matching S3 connector
from pyspark.sql import SparkSession

spark.stop()

In [ ]:
!apt-get -qq update
!apt-get -qq install -y maven

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libapache-pom-java.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../00-libapache-pom-java_29-2_all.deb ...
Unpacking libapache-pom-java (29-2) ...
Selecting previously unselected package libatinject-jsr330-api-java.
Preparing to unpack .../01-libatinject-jsr330-api-java_1.0+ds1-5_all.deb ...
Unpacking libatinject-jsr330-api-java (1.0+ds1-5) ...
Selecting previously unselected package libgeronimo-interceptor-3.0-spec-java.
Preparing to unpack .../02-libgeronimo-interceptor-3.0-spec-java_1.0.1-4fakesync_all.deb ...
Unpacking libgeronimo-interceptor-3.0-spec-java (1.0.1-4fakesync) ...
Selecting previously unselected package libcdi-api-java.
Preparing to unpack .../03-libcdi-a

In [ ]:
!mvn -version

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Apache Maven 3.8.7
Maven home: /usr/share/maven
Java version: 21.0.12.1, vendor: Ubuntu, runtime: /usr/lib/jvm/java-21-openjdk-amd64
Default locale: en_US, platform encoding: UTF-8
OS name: "linux", version: "6.6.122+", arch: "amd64", family: "unix"


In [ ]:
# Download the compatible connector
!mvn -q dependency:get -Dartifact=org.apache.hadoop:hadoop-aws:3.4.1

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [ ]:
from pathlib import Path

jar = Path.home() / ".m2/repository/org/apache/hadoop/hadoop-aws/3.4.1/hadoop-aws-3.4.1.jar"

print("S3A connector JAR exists:", jar.exists())
print("JAR path:", jar)

S3A connector JAR exists: True
JAR path: /root/.m2/repository/org/apache/hadoop/hadoop-aws/3.4.1/hadoop-aws-3.4.1.jar


In [ ]:
# Configure Spark with the S3 connector
!find ~/.m2/repository/org/apache/hadoop/ -name "*.jar" | grep -E 'hadoop-aws-3.4.1.jar|hadoop-common-3.4.1.jar'

/root/.m2/repository/org/apache/hadoop/hadoop-aws/3.4.1/hadoop-aws-3.4.1.jar
/root/.m2/repository/org/apache/hadoop/hadoop-common/3.4.1/hadoop-common-3.4.1.jar


In [ ]:
!find ~/.m2/repository/software/amazon/awssdk/ -name "*.jar" | head -20

/root/.m2/repository/software/amazon/awssdk/bundle/2.24.6/bundle-2.24.6.jar


In [ ]:
# Restart Spark with the connector
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path

hadoop_aws = str(
    Path.home() / ".m2/repository/org/apache/hadoop/hadoop-aws/3.4.1/hadoop-aws-3.4.1.jar"
)

aws_sdk = str(
    Path.home() / ".m2/repository/software/amazon/awssdk/bundle/2.24.6/bundle-2.24.6.jar"
)

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .config("spark.jars", f"{hadoop_aws},{aws_sdk}")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print("Spark version:", spark.version)
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.0.1
Hadoop version: 3.4.1


In [ ]:
jvm = spark.sparkContext._jvm

try:
    s3a_class = jvm.java.lang.Class.forName(
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    print("SUCCESS: S3A filesystem class loaded:", s3a_class.getName())
except Exception as e:
    print("S3A class loading failed:")
    print(str(e)[:2000])

S3A class loading failed:
An error occurred while calling z:java.lang.Class.forName.
: java.lang.ClassNotFoundException: org.apache.hadoop.fs.s3a.S3AFileSystem
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClass(BuiltinClassLoader.java:641)
	at java.base/jdk.internal.loader.ClassLoaders$AppClassLoader.loadClass(ClassLoaders.java:188)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:423)
	at java.base/java.lang.Class.forName(Class.java:414)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75)
	at java.base/jdk.internal.reflect.MethodAccessorImpl.invoke(MethodAccessorImpl.java:51)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:59)
	at java.base/java.lang.reflect.Method.in

In [ ]:
# Start a fresh Spark session with dependency resolution
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-V2")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.1"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)
print(
    "Hadoop version:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.0.1
Hadoop version: 3.4.1


In [ ]:
jvm = spark.sparkContext._jvm

try:
    cls = jvm.java.lang.Class.forName(
        "org.apache.hadoop.fs.s3a.S3AFileSystem",
        True,
        jvm.java.lang.Thread.currentThread().getContextClassLoader()
    )
    print("SUCCESS:", cls.getName())
except Exception as e:
    print("FAILED:", str(e)[:3000])

SUCCESS: org.apache.hadoop.fs.s3a.S3AFileSystem


In [ ]:
# Set up AWS credentials securely
import getpass
import boto3

aws_access_key_id = getpass.getpass("AWS Access Key ID: ")
aws_secret_access_key = getpass.getpass("AWS Secret Access Key: ")

session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="eu-north-1"
)

s3 = session.client("s3")

print("AWS client created.")

ModuleNotFoundError: No module named 'boto3'

In [ ]:
# install boto3
%pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.8 MB/s eta 0:00:00


In [ ]:
# Retry the AWS client setup
import getpass
import boto3

aws_access_key_id = getpass.getpass("AWS Access Key ID: ")
aws_secret_access_key = getpass.getpass("AWS Secret Access Key: ")

session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="eu-north-1"
)

s3 = session.client("s3")

print("AWS client created.")

KeyboardInterrupt: Interrupted by user

In [ ]:
%pip install boto3

In [ ]:
import getpass
import boto3

aws_access_key_id = getpass.getpass("AWS Access Key ID: ")
aws_secret_access_key = getpass.getpass("AWS Secret Access Key: ")

session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="eu-north-1"
)

s3 = session.client("s3")

print("AWS client created.")

AWS Access Key ID: ··········
AWS Secret Access Key: ··········
AWS client created.


In [ ]:
# Test read-only access
BUCKET = "amogh-iot-data-platform-2026-946486414750-eu-north-1-an"
PREFIX = "iot-platform-v2/"

response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX,
    MaxKeys=20
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} — {obj['Size']:,} bytes")

iot-platform-v2/processed/ — 0 bytes
iot-platform-v2/processed/data_quality_report.json — 445 bytes
iot-platform-v2/processed/sensor_readings_clean.parquet — 7,651,253 bytes
iot-platform-v2/quarantine/ — 0 bytes
iot-platform-v2/quarantine/sensor_readings_quarantine.csv — 138 bytes
iot-platform-v2/raw/ — 0 bytes
iot-platform-v2/raw/sensor_readings_source.csv — 36,287,827 bytes
iot-platform-v2/warehouse/iceberg/ — 0 bytes


In [ ]:
# Configure Spark to access S3

# Configure Spark's S3A connector
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.access.key", aws_access_key_id)
hadoop_conf.set("fs.s3a.secret.key", aws_secret_access_key)
hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)
hadoop_conf.set("fs.s3a.endpoint.region", "eu-north-1")

print("Spark S3 configuration completed.")

Spark S3 configuration completed.


In [ ]:
# Read the Parquet file using Spark
s3_path = (
    "s3a://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
    "iot-platform-v2/processed/sensor_readings_clean.parquet"
)

df = spark.read.parquet(s3_path)

print("Schema:")
df.printSchema()

print("First 5 records:")
df.show(5, truncate=False)

Schema:
root
 |-- source_record_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- CO2CosIRValue: double (nullable = true)
 |-- CO2MG811Value: double (nullable = true)
 |-- MOX1: double (nullable = true)
 |-- MOX2: double (nullable = true)
 |-- MOX3: double (nullable = true)
 |-- MOX4: double (nullable = true)
 |-- COValue: double (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_dataset_id: long (nullable = true)

First 5 records:
+----------------+--------------------------+-----------+--------+-------------+-------------+-----+-----+-----+-----+-------+--------------------------------+-----------------+
|source_record_id|event_time                |temperature|humidity|CO2CosIRValue|CO2MG811Value|MOX1 |MOX2 |MOX3 |MOX4 |COValue|source_dataset                  |source_dataset_id|
+----------------+--------------------------+-----------+--------+-------

In [ ]:
# Verify the full dataset with Spark
from pyspark.sql import functions as F

# 1. Count all records
total_rows = df.count()

# 2. Count unique source record IDs
unique_ids = df.select("source_record_id").distinct().count()

# 3. Check missing values in important columns
columns_to_check = [
    "source_record_id",
    "event_time",
    "temperature",
    "humidity",
    "CO2CosIRValue",
    "CO2MG811Value",
    "MOX1",
    "MOX2",
    "MOX3",
    "MOX4",
    "COValue"
]

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in columns_to_check
])

# 4. Check timestamp range
timestamp_range = df.agg(
    F.min("event_time").alias("earliest_event"),
    F.max("event_time").alias("latest_event")
)

print(f"Total records: {total_rows:,}")
print(f"Unique record IDs: {unique_ids:,}")
print(f"Duplicate IDs: {total_rows - unique_ids:,}")

print("\nMissing values:")
null_counts.show(truncate=False)

print("\nTimestamp range:")
timestamp_range.show(truncate=False)

Total records: 416,153
Unique record IDs: 416,153
Duplicate IDs: 0

Missing values:
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+
|source_record_id|event_time|temperature|humidity|CO2CosIRValue|CO2MG811Value|MOX1|MOX2|MOX3|MOX4|COValue|
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+
|0               |0         |0          |0       |0            |0            |0   |0   |0   |0   |0      |
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+


Timestamp range:
+--------------------------+--------------------------+
|earliest_event            |latest_event              |
+--------------------------+--------------------------+
|2019-11-06 10:37:13.038174|2020-02-13 10:58:14.014336|
+--------------------------+--------------------------+



In [ ]:
print("Spark version:", spark.version)

print(
    "Java version:",
    spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
)

print("Dataset records:", df.count())

Spark version: 4.0.1
Java version: 21.0.12.1
Dataset records: 416153


In [ ]:
# Install Iceberg libraries in Spark

import os

# Make the existing credentials available to AWS SDKs.
# Never print these values.
os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_access_key
os.environ["AWS_REGION"] = "eu-north-1"
os.environ["AWS_DEFAULT_REGION"] = "eu-north-1"

# Stop the current Spark session so the new libraries load at startup.
spark.stop()

print("Old Spark session stopped.")

Old Spark session stopped.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-Iceberg")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.1,"
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.11.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.11.0"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.glue_catalog",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.glue_catalog.type", "glue")
    .config(
        "spark.sql.catalog.glue_catalog.warehouse",
        "s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
        "iot-platform-v2/warehouse/iceberg/"
    )
    .config(
        "spark.sql.catalog.glue_catalog.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.glue_catalog.glue.region",
        "eu-north-1"
    )
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Iceberg Spark session initialized.")

Spark version: 4.0.1
Iceberg Spark session initialized.


In [ ]:
# test the iceberg catalog
# Check whether Spark can communicate with the Glue catalog
spark.sql("SHOW NAMESPACES IN glue_catalog").show(truncate=False)

Py4JJavaError: An error occurred while calling o215.sql.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'glue_catalog': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1830)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:56)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:469)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:56)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndNamespace$.unapply(LookupCatalog.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:92)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:38)

In [ ]:
# check whether iceberg JARs actually loaded
import os
import glob

print("Spark version:", spark.version)
print(
    "Configured packages:",
    spark.sparkContext.getConf().get("spark.jars.packages", "Not set")
)

print("\nIceberg JARs in Ivy cache:")
ivy_jars = glob.glob("/root/.ivy2/jars/*iceberg*.jar")

if ivy_jars:
    for jar in ivy_jars:
        print(jar)
else:
    print("No Iceberg JARs found in Ivy cache.")

print("\nIceberg JARs in Maven cache:")
maven_jars = glob.glob("/root/.m2/repository/org/apache/iceberg/**/*.jar",
                       recursive=True)

if maven_jars:
    for jar in maven_jars:
        print(jar)
else:
    print("No Iceberg JARs found in Maven cache.")

print("\nTesting Iceberg catalog class:")
try:
    jvm = spark.sparkContext._jvm
    jvm.java.lang.Class.forName(
        "org.apache.iceberg.spark.SparkCatalog",
        True,
        jvm.java.lang.Thread.currentThread().getContextClassLoader()
    )
    print("SUCCESS: Iceberg catalog class is available.")
except Exception as e:
    print(type(e).__name__, str(e)[:1000])

Spark version: 4.0.1
Configured packages: org.apache.hadoop:hadoop-aws:3.4.1,org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.11.0,org.apache.iceberg:iceberg-aws-bundle:1.11.0

Iceberg JARs in Ivy cache:
No Iceberg JARs found in Ivy cache.

Iceberg JARs in Maven cache:
No Iceberg JARs found in Maven cache.

Testing Iceberg catalog class:
Py4JJavaError An error occurred while calling z:java.lang.Class.forName.
: java.lang.ClassNotFoundException: org.apache.iceberg.spark.SparkCatalog
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:536)
	at java.base/java.lang.Class.forName(Class.java:515)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorI

In [ ]:
import os
import getpass

aws_access_key_id = getpass.getpass("AWS Access Key ID: ")
aws_secret_access_key = getpass.getpass("AWS Secret Access Key: ")

os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_access_key
os.environ["AWS_REGION"] = "eu-north-1"
os.environ["AWS_DEFAULT_REGION"] = "eu-north-1"

print("Credentials configured for this runtime.")

AWS Access Key ID: ··········
AWS Secret Access Key: ··········
Credentials configured for this runtime.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-Iceberg")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.1,"
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.11.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.11.0"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.glue_catalog",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.glue_catalog.type", "glue")
    .config(
        "spark.sql.catalog.glue_catalog.warehouse",
        "s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
        "iot-platform-v2/warehouse/iceberg/"
    )
    .config(
        "spark.sql.catalog.glue_catalog.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.glue_catalog.glue.region",
        "eu-north-1"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark started with Iceberg dependencies.")

Spark version: 4.0.1
Spark started with Iceberg dependencies.


In [ ]:
jvm = spark.sparkContext._jvm

jvm.java.lang.Class.forName(
    "org.apache.iceberg.spark.SparkCatalog",
    True,
    jvm.java.lang.Thread.currentThread().getContextClassLoader()
)

print("SUCCESS: Iceberg catalog class is available.")

SUCCESS: Iceberg catalog class is available.


In [ ]:
spark.sql("SHOW NAMESPACES IN glue_catalog").show(truncate=False)

+---------+
|namespace|
+---------+
+---------+



In [ ]:
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS glue_catalog.iot_platform_db
""")

spark.sql("SHOW NAMESPACES IN glue_catalog").show(truncate=False)

Py4JJavaError: An error occurred while calling o47.sql.
: software.amazon.awssdk.services.glue.model.AccessDeniedException: User: arn:aws:iam::946486414750:user/iot-python-user is not authorized to perform: glue:CreateDatabase on resource: arn:aws:glue:eu-north-1:946486414750:database/iot_platform_db because no identity-based policy allows the glue:CreateDatabase action (Service: Glue, Status Code: 400, Request ID: d77974ca-08d6-4c83-860f-8b9ea44d9a05) (SDK Attempt Count: 1)
	at software.amazon.awssdk.services.glue.model.AccessDeniedException$BuilderImpl.build(AccessDeniedException.java:150)
	at software.amazon.awssdk.services.glue.model.AccessDeniedException$BuilderImpl.build(AccessDeniedException.java:98)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.utils.RetryableStageHelper.retryPolicyDisallowedRetryException(RetryableStageHelper.java:180)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.RetryableStage.execute(RetryableStage.java:86)
	at software.amazo

In [ ]:
# retry
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS glue_catalog.iot_platform_db
""")

spark.sql("SHOW NAMESPACES IN glue_catalog").show(truncate=False)

+---------+
|namespace|
+---------+
+---------+



In [ ]:
spark.sql("""
    SHOW NAMESPACES IN glue_catalog
""").show(truncate=False)

print("Checking project database...")

try:
    spark.sql("""
        DESCRIBE NAMESPACE glue_catalog.iot_platform_db
    """).show(truncate=False)
except Exception as e:
    print("Database check failed:")
    print(str(e)[:2000])

+---------+
|namespace|
+---------+
+---------+

Checking project database...
+--------------+---------------+
|info_name     |info_value     |
+--------------+---------------+
|Catalog Name  |glue_catalog   |
|Namespace Name|iot_platform_db|
|Owner         |root           |
+--------------+---------------+



In [ ]:
# Load the Clean Sensor Data from s3
s3_path = (
    "s3a://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
    "iot-platform-v2/processed/sensor_readings_clean.parquet"
)

df = spark.read.parquet(s3_path)

print("Total records:", df.count())
print("Columns:", df.columns)

df.printSchema()
df.show(5, truncate=False)

Total records: 416153
Columns: ['source_record_id', 'event_time', 'temperature', 'humidity', 'CO2CosIRValue', 'CO2MG811Value', 'MOX1', 'MOX2', 'MOX3', 'MOX4', 'COValue', 'source_dataset', 'source_dataset_id']
root
 |-- source_record_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- CO2CosIRValue: double (nullable = true)
 |-- CO2MG811Value: double (nullable = true)
 |-- MOX1: double (nullable = true)
 |-- MOX2: double (nullable = true)
 |-- MOX3: double (nullable = true)
 |-- MOX4: double (nullable = true)
 |-- COValue: double (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_dataset_id: long (nullable = true)

+----------------+--------------------------+-----------+--------+-------------+-------------+-----+-----+-----+-----+-------+--------------------------------+-----------------+
|source_record_id|event_time                |temperature|humidity|C

In [ ]:
# Create the Iceberg Table
# Register the clean Spark DataFrame as a temporary SQL view
df.createOrReplaceTempView("sensor_readings_source")

# Create the Iceberg table in AWS Glue and store its data in S3
spark.sql("""
    CREATE TABLE glue_catalog.iot_platform_db.sensor_readings
    USING iceberg
    LOCATION 's3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/warehouse/iceberg/sensor_readings'
    AS SELECT * FROM sensor_readings_source
""")

print("Iceberg table created successfully.")

Iceberg table created successfully.


In [ ]:
# Verify the table
table_name = "glue_catalog.iot_platform_db.sensor_readings"

print("Total rows in Iceberg table:")
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {table_name}").show()

print("Sample records:")
spark.sql(f"""
    SELECT source_record_id, event_time, temperature, humidity
    FROM {table_name}
    ORDER BY source_record_id
    LIMIT 5
""").show(truncate=False)

print("Table details:")
spark.sql(f"DESCRIBE TABLE {table_name}").show(truncate=False)

Total rows in Iceberg table:
+----------+
|total_rows|
+----------+
|    416153|
+----------+

Sample records:
+----------------+--------------------------+-----------+--------+
|source_record_id|event_time                |temperature|humidity|
+----------------+--------------------------+-----------+--------+
|1               |2019-11-06 10:37:13.038174|19.48      |54.86   |
|2               |2019-11-06 10:37:32.744996|19.59      |54.23   |
|3               |2019-11-06 10:37:53.018087|19.63      |54.05   |
|4               |2019-11-06 10:38:13.093151|19.64      |53.74   |
|5               |2019-11-06 10:38:33.032832|19.67      |53.53   |
+----------------+--------------------------+-----------+--------+

Table details:
+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|source_record_id |bigint   |NULL   |
|event_time       |timestamp|NULL   |
|temperature      |double   |NULL   |
|humidity         |double   |NULL   |
|CO2

In [ ]:
# Verify the Iceberg files in S3
import boto3

bucket = "amogh-iot-data-platform-2026-946486414750-eu-north-1-an"
prefix = "iot-platform-v2/warehouse/iceberg/sensor_readings/"

s3_client = boto3.client(
    "s3",
    region_name="eu-north-1",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

response = s3_client.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix,
    MaxKeys=30
)

objects = response.get("Contents", [])

print("Objects found:", len(objects))

for obj in objects:
    print(f"{obj['Key']} | {obj['Size']:,} bytes")

Objects found: 4
iot-platform-v2/warehouse/iceberg/sensor_readings/data/00000-5-f056cfd3-e898-4429-a9ee-e1ecf5497d20-0-00001.parquet | 4,308,373 bytes
iot-platform-v2/warehouse/iceberg/sensor_readings/metadata/00000-8303753e-a967-4243-9111-97030ecb9b54.metadata.json | 2,569 bytes
iot-platform-v2/warehouse/iceberg/sensor_readings/metadata/a149a1d2-bad1-4db2-9e52-f59ec667464b-m0.avro | 8,052 bytes
iot-platform-v2/warehouse/iceberg/sensor_readings/metadata/snap-3374884901694200692-1-a149a1d2-bad1-4db2-9e52-f59ec667464b.avro | 4,496 bytes


In [ ]:
# test an actual iceberg table update
table_name = "glue_catalog.iot_platform_db.sensor_readings"

# Record the current snapshot
print("Snapshots before update:")
spark.sql(f"""
    SELECT committed_at, snapshot_id, operation
    FROM {table_name}.snapshots
""").show(truncate=False)

# Change a table property; this does not change the sensor data
spark.sql(f"""
    ALTER TABLE {table_name}
    SET TBLPROPERTIES ('project.stage' = 'iceberg-verified')
""")

# Confirm the property
spark.sql(f"SHOW TBLPROPERTIES {table_name}").show(truncate=False)

Snapshots before update:
+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-14 11:33:09.981|3374884901694200692|append   |
+-----------------------+-------------------+---------+

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |3374884901694200692|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|project.stage                  |iceberg-verified   |
|write.parquet.compression-codec|zstd               |
+-------------------------------+-------------------+



In [ ]:
# Testing real analytical query
table_name = "glue_catalog.iot_platform_db.sensor_readings"

spark.sql(f"""
    SELECT
        source_dataset,
        COUNT(*) AS total_readings,
        ROUND(AVG(temperature), 2) AS avg_temperature,
        ROUND(MIN(temperature), 2) AS min_temperature,
        ROUND(MAX(temperature), 2) AS max_temperature,
        ROUND(AVG(humidity), 2) AS avg_humidity,
        MIN(event_time) AS first_reading,
        MAX(event_time) AS last_reading
    FROM {table_name}
    GROUP BY source_dataset
""").show(truncate=False)

+--------------------------------+--------------+---------------+---------------+---------------+------------+--------------------------+--------------------------+
|source_dataset                  |total_readings|avg_temperature|min_temperature|max_temperature|avg_humidity|first_reading             |last_reading              |
+--------------------------------+--------------+---------------+---------------+---------------+------------+--------------------------+--------------------------+
|UCI_Single_Elder_Home_Monitoring|416153        |20.12          |13.53          |26.14          |52.24       |2019-11-06 10:37:13.038174|2020-02-13 10:58:14.014336|
+--------------------------------+--------------+---------------+---------------+---------------+------------+--------------------------+--------------------------+



In [ ]:
# Verify partitioning and query performance
table_name = "glue_catalog.iot_platform_db.sensor_readings"

print("TABLE PARTITIONING")
spark.sql(f"""
    SELECT *
    FROM {table_name}.partitions
""").show(truncate=False)

print("QUERY PLAN")
spark.sql(f"""
    SELECT
        DATE(event_time) AS event_date,
        COUNT(*) AS readings,
        ROUND(AVG(temperature), 2) AS avg_temperature
    FROM {table_name}
    WHERE event_time >= TIMESTAMP '2020-01-01 00:00:00'
      AND event_time <  TIMESTAMP '2020-02-01 00:00:00'
    GROUP BY DATE(event_time)
""").explain("formatted")

TABLE PARTITIONING
+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|416153      |1         |4308373                      |0                           |0                         |0                           |0                         |2026-10-14 11:33:09.981|3374884901694200692     |
+------------+----------+-----------------------------+----------------------------+--------------------------+--

In [ ]:
 # Validate time-based queries
 table_name = "glue_catalog.iot_platform_db.sensor_readings"

spark.sql(f"""
    SELECT
        DATE(event_time) AS event_date,
        COUNT(*) AS readings,
        ROUND(AVG(temperature), 2) AS avg_temperature,
        ROUND(AVG(humidity), 2) AS avg_humidity
    FROM {table_name}
    WHERE event_time >= TIMESTAMP '2020-01-01 00:00:00'
      AND event_time <  TIMESTAMP '2020-02-01 00:00:00'
    GROUP BY DATE(event_time)
    ORDER BY event_date
""").show(40, truncate=False)

spark.sql(f"""
    SELECT COUNT(*) AS january_readings
    FROM {table_name}
    WHERE event_time >= TIMESTAMP '2020-01-01 00:00:00'
      AND event_time <  TIMESTAMP '2020-02-01 00:00:00'
""").show()

+----------+--------+---------------+------------+
|event_date|readings|avg_temperature|avg_humidity|
+----------+--------+---------------+------------+
|2020-01-01|4320    |21.49          |49.05       |
|2020-01-02|4320    |21.21          |48.32       |
|2020-01-03|4320    |20.1           |49.55       |
|2020-01-04|4320    |19.29          |47.67       |
|2020-01-05|4320    |19.62          |47.77       |
|2020-01-06|4320    |18.47          |47.71       |
|2020-01-07|4321    |18.74          |47.0        |
|2020-01-08|4320    |17.85          |49.34       |
|2020-01-09|4320    |17.7           |51.45       |
|2020-01-10|4320    |18.91          |53.51       |
|2020-01-11|4320    |18.68          |50.33       |
|2020-01-12|4320    |17.04          |48.9        |
|2020-01-13|4320    |16.42          |49.23       |
|2020-01-14|4320    |16.23          |49.1        |
|2020-01-15|4320    |16.51          |49.35       |
|2020-01-16|4320    |17.66          |48.47       |
|2020-01-17|4301    |17.73     

In [ ]:
# Build incremental ingestion
# inspect the existing table's current snapshot history
spark.sql("""
    SELECT
        committed_at,
        snapshot_id,
        operation,
        summary
    FROM glue_catalog.iot_platform_db.sensor_readings.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                               

In [ ]:
# simulate incremental ingestion safely
# Create the test table
spark.sql("""
    CREATE TABLE glue_catalog.iot_platform_db.sensor_readings_incremental_test
    USING iceberg
    LOCATION 's3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/iot-platform-v2/warehouse/iceberg/sensor_readings_incremental_test'
    AS
    SELECT *
    FROM sensor_readings_source
    WHERE source_record_id BETWEEN 1 AND 10
""")

spark.sql("""
    SELECT COUNT(*) AS initial_records
    FROM glue_catalog.iot_platform_db.sensor_readings_incremental_test
""").show()

+---------------+
|initial_records|
+---------------+
|             10|
+---------------+



In [ ]:
# Append a second batch
from pyspark.sql import functions as F

test_table = "glue_catalog.iot_platform_db.sensor_readings_incremental_test"

# Select the next five records as a simulated incoming batch
new_batch = (
    spark.table("sensor_readings_source")
    .filter(F.col("source_record_id").between(11, 15))
)

print("Incoming batch record count:", new_batch.count())

# Append this batch to the separate Iceberg test table
new_batch.writeTo(test_table).append()

# Verify the table's new total
spark.sql(f"""
    SELECT COUNT(*) AS total_records
    FROM {test_table}
""").show()

Incoming batch record count: 5
+-------------+
|total_records|
+-------------+
|           15|
+-------------+



In [ ]:
# Verify Iceberg snapshot history
spark.sql("""
    SELECT
        committed_at,
        snapshot_id,
        operation,
        summary['added-records'] AS added_records,
        summary['total-records'] AS total_records,
        summary['added-data-files'] AS added_data_files
    FROM glue_catalog.iot_platform_db.sensor_readings_incremental_test.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+---------+-------------+-------------+----------------+
|committed_at           |snapshot_id        |operation|added_records|total_records|added_data_files|
+-----------------------+-------------------+---------+-------------+-------------+----------------+
|2026-10-14 11:48:34.834|3354236864556765872|append   |10           |10           |1               |
|2026-10-14 11:49:41.37 |7334089189594877055|append   |5            |15           |1               |
+-----------------------+-------------------+---------+-------------+-------------+----------------+



In [ ]:
# make ingestion safe against duplicates
# Prepare a repeated batch
retry_batch = (
    spark.table("sensor_readings_source")
    .filter("source_record_id BETWEEN 11 AND 15")
)

print("Records in retried batch:", retry_batch.count())
retry_batch.show(5, truncate=False)

Records in retried batch: 5
+----------------+--------------------------+-----------+--------+-------------+-------------+-----+-----+-----+-----+-------+--------------------------------+-----------------+
|source_record_id|event_time                |temperature|humidity|CO2CosIRValue|CO2MG811Value|MOX1 |MOX2 |MOX3 |MOX4 |COValue|source_dataset                  |source_dataset_id|
+----------------+--------------------------+-----------+--------+-------------+-------------+-----+-----+-----+-----+-------+--------------------------------+-----------------+
|11              |2019-11-06 10:40:33.085384|19.64      |53.51   |127.0        |573.0        |486.0|732.0|650.0|566.0|119.0  |UCI_Single_Elder_Home_Monitoring|799              |
|12              |2019-11-06 10:40:53.004693|19.64      |53.51   |127.0        |574.0        |487.0|732.0|650.0|566.0|119.0  |UCI_Single_Elder_Home_Monitoring|799              |
|13              |2019-11-06 10:41:13.02957 |19.66      |53.51   |127.0        |57

In [ ]:
# Run a duplicate-safe merge
retry_batch.createOrReplaceTempView("incoming_retry_batch")

spark.sql("""
    MERGE INTO glue_catalog.iot_platform_db.sensor_readings_incremental_test AS target
    USING incoming_retry_batch AS source
    ON target.source_record_id = source.source_record_id
    WHEN NOT MATCHED THEN
        INSERT *
""")

spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT source_record_id) AS unique_record_ids
    FROM glue_catalog.iot_platform_db.sensor_readings_incremental_test
""").show()

+-------------+-----------------+
|total_records|unique_record_ids|
+-------------+-----------------+
|           15|               15|
+-------------+-----------------+



In [ ]:
# validate the test table
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT source_record_id) AS unique_ids,
        MIN(source_record_id) AS first_id,
        MAX(source_record_id) AS last_id,
        ROUND(AVG(temperature), 2) AS avg_temperature
    FROM glue_catalog.iot_platform_db.sensor_readings_incremental_test
""").show()

+-------------+----------+--------+-------+---------------+
|total_records|unique_ids|first_id|last_id|avg_temperature|
+-------------+----------+--------+-------+---------------+
|           15|        15|       1|     15|          19.64|
+-------------+----------+--------+-------+---------------+



In [ ]:
# build the data-quality and analytics layer
from pyspark.sql import functions as F

main_table = "glue_catalog.iot_platform_db.sensor_readings"
main_df = spark.table(main_table)

quality_summary = main_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("source_record_id").alias("unique_ids"),
    F.sum(F.col("source_record_id").isNull().cast("int")).alias("null_ids"),
    F.sum(F.col("event_time").isNull().cast("int")).alias("null_timestamps"),
    F.sum(F.col("temperature").isNull().cast("int")).alias("null_temperatures"),
    F.sum(
        (
            (F.col("temperature") < -40) |
            (F.col("temperature") > 85)
        ).cast("int")
    ).alias("temperature_outside_check_range")
)

quality_summary.show(truncate=False)

+-------------+----------+--------+---------------+-----------------+-------------------------------+
|total_records|unique_ids|null_ids|null_timestamps|null_temperatures|temperature_outside_check_range|
+-------------+----------+--------+---------------+-----------------+-------------------------------+
|416153       |416153    |0       |0              |0                |0                              |
+-------------+----------+--------+---------------+-----------------+-------------------------------+



In [ ]:
# Comprehensive data-quality checks
from pyspark.sql import functions as F

table_name = "glue_catalog.iot_platform_db.sensor_readings"
df_quality = spark.table(table_name)

# 1. Overall counts and duplicate IDs
total_rows = df_quality.count()
unique_ids = df_quality.select("source_record_id").distinct().count()

print("=== RECORD VALIDATION ===")
print("Total records:", total_rows)
print("Unique record IDs:", unique_ids)
print("Duplicate ID excess:", total_rows - unique_ids)

# 2. Null counts for every column
null_checks = df_quality.agg(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in df_quality.columns
])

print("\n=== NULL COUNTS ===")
null_checks.show(truncate=False)

# 3. Sensor value ranges and timestamp coverage
sensor_columns = [
    "temperature", "humidity", "CO2CosIRValue",
    "CO2MG811Value", "MOX1", "MOX2", "MOX3", "MOX4", "COValue"
]

range_checks = df_quality.agg(
    F.min("event_time").alias("first_reading"),
    F.max("event_time").alias("last_reading"),
    *[
        F.min(c).alias(f"{c}_min")
        for c in sensor_columns
    ],
    *[
        F.max(c).alias(f"{c}_max")
        for c in sensor_columns
    ]
)

print("\n=== SENSOR RANGES AND TIME COVERAGE ===")
range_checks.show(truncate=False)

# 4. Count non-finite numeric values, if any
non_finite_checks = df_quality.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.isnan(c) | (F.abs(F.col(c)) == float("inf"))),
            1
        ).otherwise(0)
    ).alias(c)
    for c in sensor_columns
])

print("\n=== NON-FINITE SENSOR VALUES ===")
non_finite_checks.show(truncate=False)

=== RECORD VALIDATION ===
Total records: 416153
Unique record IDs: 416153
Duplicate ID excess: 0

=== NULL COUNTS ===
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+--------------+-----------------+
|source_record_id|event_time|temperature|humidity|CO2CosIRValue|CO2MG811Value|MOX1|MOX2|MOX3|MOX4|COValue|source_dataset|source_dataset_id|
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+--------------+-----------------+
|0               |0         |0          |0       |0            |0            |0   |0   |0   |0   |0      |0             |0                |
+----------------+----------+-----------+--------+-------------+-------------+----+----+----+----+-------+--------------+-----------------+


=== SENSOR RANGES AND TIME COVERAGE ===
+--------------------------+--------------------------+---------------+------------+-----------------+-----------------+--------+--------+--

In [5]:
# Timestamp gaps and record reconciliation
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bucket = (
    "s3a://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
    "iot-platform-v2/"
)

# Read the three pipeline stages
raw_df = (
    spark.read
    .option("header", True)
    .csv(bucket + "raw/sensor_readings_source.csv")
)

clean_df = spark.read.parquet(
    bucket + "processed/sensor_readings_clean.parquet"
)

iceberg_df = spark.table(
    "glue_catalog.iot_platform_db.sensor_readings"
)

raw_count = raw_df.count()
clean_count = clean_df.count()
iceberg_count = iceberg_df.count()

print("=== SOURCE-TO-ICEBERG RECONCILIATION ===")
print(f"Raw CSV records:       {raw_count}")
print(f"Clean Parquet records: {clean_count}")
print(f"Iceberg records:       {iceberg_count}")
print(f"Raw → Clean difference: {raw_count - clean_count}")
print(f"Clean → Iceberg difference: {clean_count - iceberg_count}")

# Measure gaps between consecutive readings
window = Window.orderBy("event_time")

gaps_df = (
    iceberg_df
    .select("event_time")
    .withColumn("previous_time", F.lag("event_time").over(window))
    .withColumn(
        "gap_seconds",
        F.col("event_time").cast("double")
        - F.col("previous_time").cast("double")
    )
    .filter(F.col("previous_time").isNotNull())
)

gap_summary = gaps_df.agg(
    F.sum((F.col("gap_seconds") > 60).cast("long"))
        .alias("gaps_over_60_seconds"),
    F.max("gap_seconds").alias("maximum_gap_seconds"),
    F.min("gap_seconds").alias("minimum_gap_seconds")
)

print("\n=== TIMESTAMP GAP ANALYSIS ===")
gap_summary.show(truncate=False)

=== SOURCE-TO-ICEBERG RECONCILIATION ===
Raw CSV records:       416153
Clean Parquet records: 416153
Iceberg records:       416153
Raw → Clean difference: 0
Clean → Iceberg difference: 0

=== TIMESTAMP GAP ANALYSIS ===
+--------------------+-------------------+-------------------+
|gaps_over_60_seconds|maximum_gap_seconds|minimum_gap_seconds|
+--------------------+-------------------+-------------------+
|31                  |166398.5602929592  |1.757328987121582  |
+--------------------+-------------------+-------------------+



In [2]:
# Reconnect the AWS credentials
import os
import getpass

aws_access_key_id = getpass.getpass("AWS Access Key ID: ")
aws_secret_access_key = getpass.getpass("AWS Secret Access Key: ")

os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_access_key
os.environ["AWS_REGION"] = "eu-north-1"
os.environ["AWS_DEFAULT_REGION"] = "eu-north-1"

print("AWS credentials configured for this runtime.")

AWS Access Key ID: ··········
AWS Secret Access Key: ··········
AWS credentials configured for this runtime.


In [3]:
# Recreate the Spark session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IoT-Platform-Iceberg")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.1,"
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.11.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.11.0"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.glue_catalog",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.glue_catalog.type", "glue")
    .config(
        "spark.sql.catalog.glue_catalog.warehouse",
        "s3://amogh-iot-data-platform-2026-946486414750-eu-north-1-an/"
        "iot-platform-v2/warehouse/iceberg/"
    )
    .config(
        "spark.sql.catalog.glue_catalog.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.glue_catalog.glue.region",
        "eu-north-1"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


In [4]:
# Verify that the main table is accessible
spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM glue_catalog.iot_platform_db.sensor_readings
""").show()

+-------------+
|total_records|
+-------------+
|       416153|
+-------------+



In [6]:
# Build daily sensor analytics
# Generate daily summaries
from pyspark.sql import functions as F

main_table = "glue_catalog.iot_platform_db.sensor_readings"

df = spark.table(main_table)

daily_analytics = (
    df
    .withColumn("reading_date", F.to_date("event_time"))
    .groupBy("reading_date")
    .agg(
        F.count("*").alias("reading_count"),

        F.round(F.avg("temperature"), 2).alias("avg_temperature"),
        F.round(F.min("temperature"), 2).alias("min_temperature"),
        F.round(F.max("temperature"), 2).alias("max_temperature"),

        F.round(F.avg("humidity"), 2).alias("avg_humidity"),

        F.round(F.avg("CO2CosIRValue"), 2).alias("avg_co2_cos_ir"),
        F.round(F.avg("CO2MG811Value"), 2).alias("avg_co2_mg811"),
        F.round(F.avg("COValue"), 2).alias("avg_co")
    )
    .orderBy("reading_date")
)

print("Number of daily summary rows:", daily_analytics.count())

daily_analytics.show(10, truncate=False)

Number of daily summary rows: 99
+------------+-------------+---------------+---------------+---------------+------------+--------------+-------------+------+
|reading_date|reading_count|avg_temperature|min_temperature|max_temperature|avg_humidity|avg_co2_cos_ir|avg_co2_mg811|avg_co|
+------------+-------------+---------------+---------------+---------------+------------+--------------+-------------+------+
|2019-11-06  |2408         |21.57          |19.0           |25.12          |51.43       |101.05        |606.75       |130.45|
|2019-11-07  |4320         |20.78          |17.74          |26.14          |52.26       |97.77         |605.88       |125.91|
|2019-11-08  |4320         |21.56          |17.53          |25.78          |49.58       |98.35         |605.12       |106.04|
|2019-11-09  |4320         |20.37          |17.62          |24.65          |50.29       |95.97         |607.15       |97.92 |
|2019-11-10  |4320         |20.28          |17.13          |25.02          |48.06    

In [7]:
# Find unusual days and overall trends
from pyspark.sql import functions as F

# 1. Overall daily temperature and humidity trends
print("=== OVERALL DAILY TRENDS ===")

daily_analytics.agg(
    F.round(F.avg("avg_temperature"), 2).alias("mean_of_daily_avg_temp"),
    F.round(F.min("avg_temperature"), 2).alias("lowest_daily_avg_temp"),
    F.round(F.max("avg_temperature"), 2).alias("highest_daily_avg_temp"),
    F.round(F.avg("avg_humidity"), 2).alias("mean_of_daily_avg_humidity"),
    F.min("reading_date").alias("first_day"),
    F.max("reading_date").alias("last_day")
).show(truncate=False)

# 2. Days with fewer than 4,000 readings
print("=== DAYS WITH FEWER THAN 4,000 READINGS ===")

daily_analytics.filter(
    F.col("reading_count") < 4000
).select(
    "reading_date",
    "reading_count",
    "avg_temperature",
    "avg_humidity"
).orderBy("reading_date").show(100, truncate=False)

# 3. Top 10 days by average CO sensor reading
print("=== TOP 10 DAYS BY AVG CO SENSOR READING ===")

daily_analytics.orderBy(
    F.desc("avg_co")
).select(
    "reading_date",
    "reading_count",
    "avg_co",
    "avg_temperature",
    "avg_humidity"
).show(10, truncate=False)

=== OVERALL DAILY TRENDS ===
+----------------------+---------------------+----------------------+--------------------------+----------+----------+
|mean_of_daily_avg_temp|lowest_daily_avg_temp|highest_daily_avg_temp|mean_of_daily_avg_humidity|first_day |last_day  |
+----------------------+---------------------+----------------------+--------------------------+----------+----------+
|20.15                 |16.23                |23.13                 |52.33                     |2019-11-06|2020-02-13|
+----------------------+---------------------+----------------------+--------------------------+----------+----------+

=== DAYS WITH FEWER THAN 4,000 READINGS ===
+------------+-------------+---------------+------------+
|reading_date|reading_count|avg_temperature|avg_humidity|
+------------+-------------+---------------+------------+
|2019-11-06  |2408         |21.57          |51.43       |
|2019-12-08  |3425         |21.55          |53.82       |
|2019-12-09  |2423         |23.13        

In [8]:
# final check: identify the top temperature and humidity days
print("=== TOP 5 DAYS BY AVERAGE TEMPERATURE ===")

daily_analytics.orderBy(
    F.desc("avg_temperature")
).select(
    "reading_date", "reading_count",
    "avg_temperature", "avg_humidity"
).show(5, truncate=False)

print("=== TOP 5 DAYS BY AVERAGE HUMIDITY ===")

daily_analytics.orderBy(
    F.desc("avg_humidity")
).select(
    "reading_date", "reading_count",
    "avg_temperature", "avg_humidity"
).show(5, truncate=False)

=== TOP 5 DAYS BY AVERAGE TEMPERATURE ===
+------------+-------------+---------------+------------+
|reading_date|reading_count|avg_temperature|avg_humidity|
+------------+-------------+---------------+------------+
|2019-12-09  |2423         |23.13          |50.48       |
|2019-12-16  |4312         |22.16          |51.54       |
|2019-12-25  |4320         |22.13          |57.92       |
|2019-12-05  |4320         |22.08          |55.94       |
|2019-12-04  |4319         |21.82          |53.44       |
+------------+-------------+---------------+------------+
only showing top 5 rows
=== TOP 5 DAYS BY AVERAGE HUMIDITY ===
+------------+-------------+---------------+------------+
|reading_date|reading_count|avg_temperature|avg_humidity|
+------------+-------------+---------------+------------+
|2020-02-04  |2254         |21.46          |59.97       |
|2020-01-23  |4320         |19.86          |59.57       |
|2020-02-11  |4320         |20.76          |58.65       |
|2020-01-26  |4115       

In [9]:
# Phase 9 — Make the pipeline repeatable
# Create the reusable pipeline function
from pyspark.sql import functions as F

def run_iot_analytics_pipeline(spark):
    table_name = "glue_catalog.iot_platform_db.sensor_readings"

    # Read the existing Iceberg table
    df = spark.table(table_name)

    # Basic validation
    total_records = df.count()
    unique_ids = df.select("source_record_id").distinct().count()

    if total_records != unique_ids:
        raise ValueError(
            f"Duplicate or null record IDs detected: "
            f"{total_records} rows, {unique_ids} unique IDs"
        )

    # Generate daily analytics
    daily_summary = (
        df.withColumn("reading_date", F.to_date("event_time"))
        .groupBy("reading_date")
        .agg(
            F.count("*").alias("reading_count"),
            F.round(F.avg("temperature"), 2).alias("avg_temperature"),
            F.round(F.min("temperature"), 2).alias("min_temperature"),
            F.round(F.max("temperature"), 2).alias("max_temperature"),
            F.round(F.avg("humidity"), 2).alias("avg_humidity"),
            F.round(F.avg("CO2CosIRValue"), 2).alias("avg_co2_cos_ir"),
            F.round(F.avg("CO2MG811Value"), 2).alias("avg_co2_mg811"),
            F.round(F.avg("COValue"), 2).alias("avg_co")
        )
        .orderBy("reading_date")
    )

    print("Pipeline validation passed")
    print("Total Iceberg records:", total_records)
    print("Unique record IDs:", unique_ids)
    print("Daily summary rows:", daily_summary.count())

    return daily_summary


daily_analytics = run_iot_analytics_pipeline(spark)

daily_analytics.show(5, truncate=False)

Pipeline validation passed
Total Iceberg records: 416153
Unique record IDs: 416153
Daily summary rows: 99
+------------+-------------+---------------+---------------+---------------+------------+--------------+-------------+------+
|reading_date|reading_count|avg_temperature|min_temperature|max_temperature|avg_humidity|avg_co2_cos_ir|avg_co2_mg811|avg_co|
+------------+-------------+---------------+---------------+---------------+------------+--------------+-------------+------+
|2019-11-06  |2408         |21.57          |19.0           |25.12          |51.43       |101.05        |606.75       |130.45|
|2019-11-07  |4320         |20.78          |17.74          |26.14          |52.26       |97.77         |605.88       |125.91|
|2019-11-08  |4320         |21.56          |17.53          |25.78          |49.58       |98.35         |605.12       |106.04|
|2019-11-09  |4320         |20.37          |17.62          |24.65          |50.29       |95.97         |607.15       |97.92 |
|2019-11-10 

In [10]:
# Add failure checks
from pyspark.sql import functions as F

def validate_iot_table(spark):
    table_name = "glue_catalog.iot_platform_db.sensor_readings"
    df = spark.table(table_name)

    checks = df.agg(
        F.count("*").alias("total_records"),
        F.countDistinct("source_record_id").alias("unique_ids"),
        F.sum(F.col("source_record_id").isNull().cast("long")).alias("null_ids"),
        F.sum(F.col("event_time").isNull().cast("long")).alias("null_timestamps"),
        F.sum(F.col("temperature").isNull().cast("long")).alias("null_temperatures"),
        F.sum(F.col("humidity").isNull().cast("long")).alias("null_humidity")
    ).first()

    print("=== PIPELINE VALIDATION ===")
    print("Total records:", checks["total_records"])
    print("Unique IDs:", checks["unique_ids"])
    print("Null IDs:", checks["null_ids"])
    print("Null timestamps:", checks["null_timestamps"])
    print("Null temperatures:", checks["null_temperatures"])
    print("Null humidity:", checks["null_humidity"])

    if (
        checks["total_records"] != checks["unique_ids"]
        or checks["null_ids"] != 0
        or checks["null_timestamps"] != 0
        or checks["null_temperatures"] != 0
        or checks["null_humidity"] != 0
    ):
        raise ValueError("Pipeline validation failed!")

    print("RESULT: PASS")
    return True


validate_iot_table(spark)

=== PIPELINE VALIDATION ===
Total records: 416153
Unique IDs: 416153
Null IDs: 0
Null timestamps: 0
Null temperatures: 0
Null humidity: 0
RESULT: PASS


True